# Market Breath Backtest

In [ ]:
import logging
import yfinance as yf
import pandas as pd
from IPython.display import display, HTML
from tabulate import tabulate

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
logger.propagate = True


pd.set_option('display.max_rows', 100)  # Adjust the number as needed
# Display the DataFrame in a scrollable window
def display_scrollable_dataframe(df, max_height=300):
    display(HTML(f'<div style="max-height: {max_height}px; overflow-y: auto;">{df.to_html()}</div>'))

def tabulate_printer(dic):
    return "\n{}".format(tabulate(
        [[k,v] for k, v in dic.items()], 
        headers=['Key', 'Value'], tablefmt='grid'))


In [ ]:
import pandas as pd
import numpy as np
import os
from tabulate import tabulate

def tabulate_printer(dic):
    return "\n{}".format(tabulate(
        [[k,v] for k, v in dic.items()], 
        headers=['Key', 'Value'], tablefmt='grid'))

def loader(ticker='^GSPC', start='1970-01-01', end='2024-05-15'):
    file_path = '/Users/bichengwang/Documents/codes/python-notebook/data/finance/{} {} {}.csv'.format(
        ticker.upper(),
        start,
        end,
    )
    if os.path.exists(file_path):
        print("File exists!")
        df = pd.read_csv(file_path, parse_dates=['Date'])
        df['Close'] = df['Adj Close']
        return df
    # Download historical data for S&P 500
    df = yf.download(ticker, start=start, end=end)
    # Save the data to a CSV file
    df.to_csv(file_path)
    df = pd.read_csv(file_path, parse_dates=['Date'])
    df['Close'] = df['Adj Close']
    return df

In [ ]:
spx = loader()
# Now df contains your data with the 'date' column parsed as datetime objects
spx.head()

In [ ]:
import matplotlib.pyplot as plt

def plot(ticker_df):
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].plot(ticker_df['Close'])
    axs[0].set_yscale('log')
    axs[0].set_title("Close History")
    axs[0].legend()
    axs[0].grid(True)
    axs[1].plot(ticker_df['Adj Close'].values)
    axs[1].set_yscale('log')
    axs[1].set_title("Adj Close")
    axs[1].legend()
    axs[1].grid(True)
    axs[2].plot(ticker_df['Volume'])
    axs[2].set_yscale('log')
    axs[2].set_title("Volume")
    axs[2].legend()
    axs[2].grid(True)
    # Adjust layout to prevent overlap
    plt.tight_layout()
    # Show the plot
    plt.show()

plot(spx)

In [ ]:
import sys
from importlib import reload

sys.path.append('/Users/bichengwang/Documents/codes/python-notebook/src')
reload(sys)

from finance.strategies.lib.indicators.td9 import calculate_td_sequential
df = loader()
df = calculate_td_sequential(df, td_val=9)

In [ ]:
import numpy as np

def exponential_mapping(input_values, min_val, max_val):
    # Generate an array of input values
    input_values = np.array(input_values)
    
    # Calculate the exponential mapping
    mapped_values = min_val + (max_val - min_val) * (np.exp(input_values) - np.exp(input_values.min())) / (np.exp(input_values.max()) - np.exp(input_values.min()))
    
    return mapped_values

# Example input values and output range
input_values = np.arange(8)  # 1, 2, 3, ..., 10
min_val = 1.
max_val = 5.

# Apply exponential mapping
output_values = exponential_mapping(input_values, min_val, max_val)

print("Input values:", input_values)
print("Output values:", output_values)

In [ ]:
from finance.strategies.lib.cash_flow_tools import generate_cash_flow
from datetime import datetime


def portfolio_history_plot(portfolio_history):
    fig, axs = plt.subplots(2, 3, figsize=(15, 10))
    axs[0][0].plot(portfolio_history['date'], portfolio_history['asset'])
    axs[0][0].set_yscale('log')
    axs[0][0].xaxis.set_major_locator(plt.matplotlib.dates.DayLocator(interval=365*5))
    axs[0][0].tick_params(axis='x', rotation=45)
    axs[0][0].set_title("Asset History")
    axs[0][0].legend()
    axs[0][0].grid(True)

    axs[0][1].plot(portfolio_history['date'], portfolio_history['share'])
    axs[0][1].set_yscale('log')
    axs[0][1].xaxis.set_major_locator(plt.matplotlib.dates.DayLocator(interval=365*5))
    axs[0][1].tick_params(axis='x', rotation=45)
    axs[0][1].set_title("Share History")
    axs[0][1].legend()
    axs[0][1].grid(True)

    axs[0][2].plot(portfolio_history['date'], portfolio_history['margin'])
    axs[0][2].xaxis.set_major_locator(plt.matplotlib.dates.DayLocator(interval=365*5))
    axs[0][2].tick_params(axis='x', rotation=45)
    axs[0][2].set_title("Margin History")
    axs[0][2].legend()
    axs[0][2].grid(True)

    axs[1][0].plot(portfolio_history['date'], portfolio_history['withdraw'])
    axs[1][0].xaxis.set_major_locator(plt.matplotlib.dates.DayLocator(interval=365*5))
    axs[1][0].tick_params(axis='x', rotation=45)
    axs[1][0].set_title("Withdraw")
    axs[1][0].legend()
    axs[1][0].grid(True)


    axs[1][1].plot(portfolio_history['date'], portfolio_history['min'])
    axs[1][1].plot(portfolio_history['date'], portfolio_history['max'])
    axs[1][1].xaxis.set_major_locator(plt.matplotlib.dates.DayLocator(interval=365*5))
    axs[1][1].tick_params(axis='x', rotation=45)
    axs[1][1].set_yscale('log')
    axs[1][1].set_title("Min Max")
    axs[1][1].legend()
    axs[1][1].grid(True)

    plt.tight_layout()
    plt.show()

def update_portfolio(portfolio, leverage, price):
    """
    loan allow negative
    lever 0.8 -> 
    loan = -0.2* asset 
    share = 0.8 * asset/price
    """
    loan, cash, share, margin = portfolio['loan'], portfolio['cash'], portfolio['share'], portfolio['margin']
    asset = cash + share * price - loan
    loan = asset * (leverage - 1.)
    share = asset * leverage / price
    cash = 0.
    margin = loan / asset

    portfolio['loan'] = loan
    portfolio['cash'] = cash
    portfolio['share'] = share
    portfolio['margin'] = margin
    portfolio['asset'] = asset
    if asset < portfolio['max']:
        portfolio['min'] = asset
    elif asset >= portfolio['max']:
        portfolio['max'] = asset
        portfolio['min'] = asset
    portfolio['withdraw'] = min(0., -1.+portfolio['min']/portfolio['max'])
    return portfolio


def passive_update_portfolio(portfolio, leverage, price):
    loan, cash, share, margin = portfolio['loan'], portfolio['cash'], portfolio['share'], portfolio['margin']
    asset = cash + share * price - loan
    if leverage - 1. <= loan / asset:
        if loan < cash:
            loan = 0.
            share += (cash - loan) / price
            cash = 0.
        else:
            loan -= cash
            share += 0.
            cash = 0.
    else:
        loan = asset * (leverage - 1.)
        share = asset * leverage / price
        cash = 0.
    margin = loan / asset
    
    portfolio['asset'] = asset
    portfolio['loan'] = loan
    portfolio['share'] = share
    portfolio['cash'] = cash
    portfolio['margin'] = margin

    if asset < portfolio['max']:
        portfolio['min'] = asset
    elif asset >= portfolio['max']:
        portfolio['max'] = asset
        portfolio['min'] = asset
    portfolio['withdraw'] = min(0., -1.+portfolio['min']/portfolio['max'])
    return portfolio


def leverage(curr, min_leverage, max_leverage):
    comb = 2**(len(curr))
    leverage = np.arange(8) * ((max_leverage - min_leverage)/(comb-1.)) + min_leverage
    return {
        "uuu":leverage[7],
        "duu":leverage[6],
        "ddu":leverage[5],
        "udu":leverage[4],
        "dud":leverage[3],
        "uud":leverage[2],
        "udd":leverage[1],
        "ddd":leverage[0],
    }[curr]


def td9_leverage(df, cash_flow, max_leverage=3., min_leverage=1., annual_interest_rate=0.05):
    df = calculate_td_sequential(df, td_val=9)
    cash = iter(cash_flow)
    initial = next(cash)
    portfolio_history = pd.DataFrame(columns=['date', 'cash', 'loan', 'share', 'margin','td9', 'asset', 'min', 'max', 'withdraw'])
    td_history = "udd"
    portfolio = {
        "date": None,
        "asset": initial,
        "cash": initial,
        "loan": 0.,
        "share": 0.,
        "margin": 0.,
        "td9": td_history,
        "min": initial,
        "max": initial,
        "withdraw": 0.0,
    }
    for _, row in df.iterrows():
        if row['trend_up']:
            td_history = td_history[1:] + "u"
        if row['trend_down']:
            td_history = td_history[1:] + "d"
        portfolio['date'] = row['Date']
        portfolio['td9'] = td_history
        leverage_ratio = leverage(td_history, min_leverage, max_leverage)
        portfolio = update_portfolio(portfolio, leverage_ratio, row['Close'])
        if row['Date'].weekday() == 0:
            portfolio_history = portfolio_history._append(portfolio.copy(), ignore_index=True)
            portfolio['cash'] += next(cash)
            portfolio['loan'] *= (1 + annual_interest_rate) ** (1. / 52.)
        
    return portfolio_history


def td9_never_loss_leverage(df, cash_flow, max_leverage=3., min_leverage=1., annual_interest_rate=0.05):
    df = calculate_td_sequential(df, td_val=9)
    cash = iter(cash_flow)
    initial = next(cash)
    portfolio_history = pd.DataFrame(columns=['date', 'cash', 'loan', 'share', 'margin','td9', 'asset', 'min', 'max', 'withdraw'])
    td_history = "udd"
    portfolio = {
        "date": "",
        "asset": initial,
        "cash": initial,
        "loan": 0.,
        "share": 0.,
        "margin": 0.,
        "td9": td_history,
        "min": initial,
        "max": initial,
        "withdraw": 0.0,
    }
    for _, row in df.iterrows():
        if row['trend_up']:
            td_history = td_history[1:] + "u"
        if row['trend_down']:
            td_history = td_history[1:] + "d"
        portfolio['date'] = row['Date']
        portfolio['td9'] = td_history
        # keep leverage if asset under previous max
        leverage_ratio = leverage(td_history, min_leverage, max_leverage)
        if portfolio['asset'] < portfolio['max']:
            leverage_ratio = max(leverage_ratio, portfolio['margin'] + 1.)
        portfolio = update_portfolio(portfolio, leverage_ratio, row['Close'])
        if row['Date'].weekday() == 0:
            portfolio_history = portfolio_history._append(portfolio.copy(), ignore_index=True)
            portfolio['cash'] += next(cash)
            portfolio['loan'] *= (1 + annual_interest_rate) ** (1. / 52.)

    return portfolio_history


def td9_passive_leverage(df, cash_flow, max_leverage=3., min_leverage=1., annual_interest_rate=0.05):
    df = calculate_td_sequential(df, td_val=9)
    cash = iter(cash_flow)
    initial = next(cash)
    portfolio_history = pd.DataFrame(columns=['date', 'cash', 'loan', 'share', 'margin','td9', 'asset', 'min', 'max', 'withdraw'])
    td_history = "udd"
    portfolio = {
        "date": "",
        "asset": initial,
        "cash": initial,
        "loan": 0.,
        "share": 0.,
        "margin": 0.,
        "td9": td_history,
        "min": initial,
        "max": initial,
        "withdraw": 0.0,
    }
    for _, row in df.iterrows():
        if row['trend_up']:
            td_history = td_history[1:] + "u"
        if row['trend_down']:
            td_history = td_history[1:] + "d"
        portfolio['date'] = row['Date']
        portfolio['td9'] = td_history

        # keep leverage if asset under previous max
        leverage_ratio = leverage(td_history, min_leverage, max_leverage)

        portfolio = passive_update_portfolio(portfolio, leverage_ratio, row['Close'])

        if row['Date'].weekday() == 0:
            portfolio_history = portfolio_history._append(portfolio.copy(), ignore_index=True)
            portfolio['cash'] += next(cash)
            portfolio['loan'] *= (1 + annual_interest_rate) ** (1. / 52.)

    return portfolio_history


def buy_and_hold(df, cash_flow, min_leverage=1., max_leverage=3., annual_interest_rate=0.05):
    cash = iter(cash_flow)
    initial = next(cash)
    portfolio_history = pd.DataFrame(columns=['date', 'cash', 'loan', 'share', 'margin','td9', 'asset', 'min', 'max', 'withdraw'])
    td_history = "ddd"
    portfolio = {
        "date": None,
        "asset": initial,
        "cash": initial,
        "loan": 0.,
        "share": 0.,
        "margin": 0.,
        "td9": td_history,
        "min": initial,
        "max": initial,
        "withdraw": 0.0,
    }
    for _, row in df.iterrows():
        portfolio['date'] = row['Date']
        portfolio['td9'] = td_history
        leverage_ratio = (min_leverage + max_leverage) / 2.
        portfolio = update_portfolio(portfolio, leverage_ratio, row['Close'])
        if row['Date'].weekday() == 0:
            portfolio_history = portfolio_history._append(portfolio.copy(), ignore_index=True)
            portfolio['cash'] += next(cash)
            portfolio['loan'] *= (1 + annual_interest_rate) ** (1. / 52.)
    return portfolio_history

def leverage_investment(
        df, 
        initial, 
        annual_investment, 
        weeks, 
        min_leverage = 1., 
        max_leverage = 2.,
        max_margin = 3., 
        annual_risk_free_rate=0.0, 
        risk_adj=0.0, 
        annual_inflation_rate = 0.0, 
        should_plot=False,
        strategy="td9_leverage"):
    df = df.tail((weeks - 1) * 5).reset_index(drop=True)
    df = calculate_td_sequential(df, td_val=9)
    cash_flow, pv, fv = generate_cash_flow(initial, annual_investment, years=weeks/52., annual_growth_rate=annual_inflation_rate, annual_risk_free_rate=annual_risk_free_rate, mode='weekly')

    portfolio_history = {
        "td9_leverage":td9_leverage, 
        "td9_never_loss_leverage":td9_never_loss_leverage,
        "td9_passive_leverage": td9_passive_leverage,
        "buy_and_hold": buy_and_hold,
    }[strategy](df, cash_flow, min_leverage=min_leverage, max_leverage=max_leverage, annual_interest_rate=(annual_risk_free_rate+risk_adj))

    portfolio_history_metrics = portfolio_history.iloc[-1].to_dict()
    portfolio_history_metrics.update({
        "pv": pv,
        "fv": fv,
        "asset vs. pv": portfolio_history.iloc[-1]['asset'] / pv,
        "asset vs. fv": portfolio_history.iloc[-1]['asset'] / fv,
    })
    if should_plot:
        portfolio_history_plot(portfolio_history)
    return portfolio_history, portfolio_history_metrics

In [ ]:
years = 10
df = loader()
df = df[(df['Date'] <= '2024-05-15')&(df['Date'] >= '1970-01-01')]
initial = 1000.
annual_investment=120.
min_leverage=1.
max_leverage=2.
max_margin=10.
annual_risk_free_rate=0.0
risk_adj=0.0
annual_inflation_rate=0.0


In [ ]:
leverage_investment(
    df,
    initial=initial, 
    annual_investment=annual_investment, 
    weeks = 52 * years, 
    min_leverage=min_leverage,
    max_leverage=max_leverage,
    max_margin=max_margin, 
    annual_risk_free_rate=annual_risk_free_rate,
    risk_adj=risk_adj,
    annual_inflation_rate=annual_inflation_rate,
    strategy="td9_leverage",
    should_plot=False,
)

In [ ]:
leverage_investment(
    df,
    initial = initial, 
    annual_investment=annual_investment, 
    weeks = 52 * years, 
    min_leverage=min_leverage,
    max_leverage=max_leverage,
    max_margin=max_margin, 
    annual_risk_free_rate=annual_risk_free_rate,
    risk_adj=risk_adj,
    annual_inflation_rate=annual_inflation_rate,
    strategy="td9_never_loss_leverage",
    should_plot=False,
)

In [ ]:
import itertools


df = loader()
df = df[(df['Date'] <= '2024-05-15')&(df['Date'] >= '1970-01-01')]

# Define the grid search grid
grid = {
    'weeks': [52*30],
    'initial': [1000.],
    'annual_investment': [120.],
    'min_leverage': [0.8, 1.],
    'max_leverage': [1.6, 2.],
    'max_margin': [10.],
    'annual_risk_free_rate': [ 0.03],
    'risk_adj': [0.02],
    'annual_inflation_rate': [0.02],
    'strategy': ["td9_leverage", "td9_never_loss_leverage", "buy_and_hold"],
    'should_plot': [True],
}

# Perform grid search
param_combinations = list(itertools.product(*grid.values()))

records = dict()
# Print the parameter combinations
for i, params in enumerate(param_combinations):
    param_dict = dict(zip(grid.keys(), params))
    portfolio_history, portfolio_history_metrics = leverage_investment(df, **param_dict)
    records[i] = param_dict.copy()
    records[i].update(portfolio_history_metrics)
    logger.info(
        tabulate_printer(records[i])
    )
    # records[i]["portfolio_history"] = portfolio_history

In [ ]:
records

In [ ]:
import itertools

# Define the grid search grid
grid = {
    'weeks': [26*52],
    'initial': [1000.],
    'annual_investment': [120.],
    'min_leverage': [1.],
    'max_leverage': [1.5, 2.],
    'max_margin': [10.],
    'annual_risk_free_rate': [ 0.04],
    'risk_adj': [0.0],
    'annual_inflation_rate': [0.0],
    'strategy': ["td9_leverage", "td9_never_loss_leverage", "td9_passive_leverage"],
    'should_plot': [True],
}

# Perform grid search
param_combinations = list(itertools.product(*grid.values()))

# Print the parameter combinations
for params in param_combinations:
    param_dict = dict(zip(grid.keys(), params))
    logger.info(tabulate_printer(param_dict))
    df = loader("NVDA", start="1999-01-22", end="2024-05-19")
    leverage_investment(df, **param_dict)

In [ ]:
import itertools

# Define the grid search grid
grid = {
    'weeks': [30*52],
    'initial': [1000.],
    'annual_investment': [120.],
    'min_leverage': [1.],
    'max_leverage': [1.5, 2.],
    'max_margin': [10.],
    'annual_risk_free_rate': [ 0.04],
    'risk_adj': [0.0],
    'annual_inflation_rate': [0.0],
    'strategy': ["td9_leverage", "td9_never_loss_leverage", "td9_passive_leverage"],
    'should_plot': [True],
}

# Perform grid search
param_combinations = list(itertools.product(*grid.values()))

# Print the parameter combinations
for params in param_combinations:
    param_dict = dict(zip(grid.keys(), params))
    logger.info(tabulate_printer(param_dict))
    df = loader("^NDX")
    leverage_investment(df, **param_dict)

In [ ]:
import itertools

# Define the grid search grid
grid = {
    'weeks': [30*52],
    'initial': [1000.],
    'annual_investment': [120.],
    'min_leverage': [1.],
    'max_leverage': [1.5],
    'max_margin': [10.],
    'annual_risk_free_rate': [ 0.04],
    'risk_adj': [0.0],
    'annual_inflation_rate': [0.0],
    'strategy': ["td9_leverage", "td9_never_loss_leverage", "td9_passive_leverage"],
    'should_plot': [True],
}

# Perform grid search
param_combinations = list(itertools.product(*grid.values()))

# Print the parameter combinations
for params in param_combinations:
    param_dict = dict(zip(grid.keys(), params))
    logger.info(tabulate_printer(param_dict))
    df = loader("TQQQ")
    leverage_investment(df, **param_dict)